In [1]:
def preprocess_titanic(df, medians=None, embarked_mode=None):
    #make copy of df
    df = df.copy()

    #extract title
    df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
    
    df['Title'] = df['Title'].replace(['Lady', 'the Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')

    #create columns for each title
    title_dummies = pd.get_dummies(df['Title'], prefix='Title', dtype=int)
    df = pd.concat([df, title_dummies], axis=1)

    #getting median value for missing values (age, fare)
    if medians is None:
        medians = {
            'Miss': df.loc[df['Title'] == 'Miss', 'Age'].median(),
            'Other': df.loc[df['Title'] != 'Miss', 'Age'].median(),
            'Fare': df['Fare'].median()
        }

    #fill in missing value (age)
    df.loc[(df['Age'].isna()) & (df['Title'] == 'Miss'), 'Age'] = medians['Miss']
    df.loc[(df['Age'].isna()) & (df['Title'] != 'Miss'), 'Age'] = medians['Other']
    
    #fill in missing value (fare)
    df['Fare'] = df['Fare'].fillna(medians['Fare'])

    #feature engineering
    df['isChild'] = ((df['Age'] < 16) | (df['Title'] == 'Master')).astype(int)
    #df['isAdult'] = 1 - df['isChild']
    #df['isElderly'] = (df['Age'] >= 60).astype(int)
    df['HasCabin'] = df['Cabin'].notna().astype(int)
    df['Sex_num'] = df['Sex'].map({'female': 1, 'male': 0})
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['FarePerPerson'] = df['Fare'] / df['FamilySize']
    #if woman in 1st class -> 3 points
    df['SexClass'] = df['Sex_num'] * (4 - df['Pclass'])
    df['FareClass'] = df['Fare'] / df['Pclass']
    df['isMother'] = ((df['Title'] == 'Mrs') & (df['Parch'] >= 1)).astype(int)
    #how many people have same ticket
    df['TicketGroupCount'] = df.groupby('Ticket')['Ticket'].transform('count')
    df['isSharedTicket'] = (df['TicketGroupCount'] > 1).astype(int)
    df['isAlone'] = (df['isSharedTicket'] == 0).astype(int)
    #df['isSmallFamily'] = ((df['FamilySize'] > 1) & (df['FamilySize'] < 5)).astype(int)
    #df['SiblingOne'] = (df['SibSp'] == 1).astype(int)
    

    #handling cabin letter
    #df['Deck'] = df['Cabin'].str[0].fillna('U').replace('T', 'Unknown')
    #df['isUpperDeck'] = df['Deck'].isin(['A', 'B', 'C']).astype(int)
    #df['isLowerDeck'] = df['Deck'].isin(['F', 'G']).astype(int)
    
    df['LastName'] = df['Name'].apply(lambda x: x.split(',')[0])
    df['GroupID'] = df['LastName'] + "_" + df['Ticket'].astype(str)
    df['MotherChild'] = (df["isMother"] == 1) | (df["isChild"]== 1).astype(int)
    #1/0
    df['PriorityGroup'] = df.groupby('GroupID')['MotherChild'].transform('max')
    #0~8..
    df['PriorityGroupCount'] = df.groupby('GroupID')['MotherChild'].transform('sum')
    df['isMaleOnlyGroup'] = ((df['PriorityGroup'] == 0) & (df['isSharedTicket'] == 1)).astype(int)

    
    df['CabinLetter'] = df['Cabin'].str[0].fillna('Unknown').replace('T', 'Unknown')
    cabin_dummies = pd.get_dummies(df['CabinLetter'], prefix='Cabin', dtype=int)
    df = pd.concat([df, cabin_dummies], axis=1)
    
    #getting value for missing value (embarked)
    if embarked_mode is None:
        embarked_mode = df['Embarked'].mode()[0]

    #filling in missing value (embarked)
    df['Embarked'] = df['Embarked'].fillna(embarked_mode)
    emb_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked', drop_first=True, dtype=int)
    df = pd.concat([df, emb_dummies], axis=1)

    #cleaning duplicates and unnecessary columns
    df = df.loc[:, ~df.columns.duplicated()]
    if 'Cabin_Unknown' in df.columns:
        df.drop('Cabin_Unknown', axis=1, inplace=True)
    
    return df, medians, embarked_mode

In [2]:
#Import
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from xgboost import XGBRegressor
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier

#load data
train = pd.read_csv('/kaggle/input/titanic/train.csv')


#process data and create columns
train, train_stats, train_mode = preprocess_titanic(train)

#target
y = train['Survived']

#features
features = [
    "Pclass", "Age", "FarePerPerson", "SibSp", "Parch",
    "Embarked_Q", "Embarked_S",
    "Cabin_A", "Cabin_B", "Cabin_C","Cabin_D", "Cabin_E", "Cabin_F","Cabin_G", "HasCabin",
    "Sex_num", "isChild", "SexClass", "isMother", "FareClass",
    "Title_Master","Title_Miss", "Title_Mr", "Title_Mrs", "Title_Rare",
    "isSharedTicket", "isMaleOnlyGroup", "PriorityGroupCount", "PriorityGroup",
]

X = train[features]

xgb_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        gamma=1,
        min_child_weight=10
    ))
])

scores = cross_val_score(
    xgb_pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)


#Accuracy: 0.8383905592869247

print("Accuracy:", scores.mean())
train.columns

Accuracy: 0.8170610758897746


Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'Title', 'Title_Master',
       'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'isChild',
       'HasCabin', 'Sex_num', 'FamilySize', 'FarePerPerson', 'SexClass',
       'FareClass', 'isMother', 'TicketGroupCount', 'isSharedTicket',
       'isAlone', 'LastName', 'GroupID', 'MotherChild', 'PriorityGroup',
       'PriorityGroupCount', 'isMaleOnlyGroup', 'CabinLetter', 'Cabin_A',
       'Cabin_B', 'Cabin_C', 'Cabin_D', 'Cabin_E', 'Cabin_F', 'Cabin_G',
       'Embarked_Q', 'Embarked_S'],
      dtype='object')

In [3]:
rf_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        #n_estimators=274,
        #random_state=23
        n_estimators=300,
        max_depth=3,
        min_samples_leaf=5,
        min_samples_split=5,
        random_state=42
    ))
])

scores = cross_val_score(
    rf_pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)


#Accuracy: 0.8293892411022534
print("Accuracy:", scores.mean())

Accuracy: 0.8081036971941498


In [4]:
#fit pipeline
modelXGB = xgb_pipeline.fit(X, y)
modelRF = rf_pipeline.fit(X, y)

test_df = pd.read_csv('/kaggle/input/titanic/test.csv')

#use the medians before
test_processed, _, _ = preprocess_titanic(test_df, medians=train_stats, embarked_mode=train_mode)

X_test = test_processed.reindex(columns=features, fill_value=0)

""""
predsXGB = modelXGB.predict(X_test)
predsRF = modelRF.predict(X_test)

final_preds = ((predsRF * 0.5 + predsXGB * 0.5) > 0.5).astype(int)
"""
probsXGB = modelXGB.predict_proba(X_test)[:, 1]
probsRF = modelRF.predict_proba(X_test)[:, 1]

final_probs = (probsRF * 0.6 + probsXGB * 0.4)
final_preds = (final_probs > 0.5).astype(int)

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": final_preds
})

submission.to_csv("submission.csv", index=False)
print("Success! Submission file created.")

Success! Submission file created.
